## Import Data

In [5]:
import numpy as np
import pandas as pd
import os
import sys

sys.path.append(os.getcwd() + '/utils')
import eda
import features
import params
import perf
import plot
import transform
from transform import LotFrontageByNeighborhoodImputer, PresenceIndicatorAdder

from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectFromModel
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

import xgboost

median_imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()

DF = {}


## Functions

In [ ]:
def load_data():
    DF['train'] = {}
    DF['train']['src'] = pd.read_csv('./kaggle/house_prices/train.csv')
    DF['test'] = {}
    DF['test']['src'] = pd.read_csv('./kaggle/house_prices/test.csv')


    DF['train']['id'] = DF['train']['src']['Id'].copy()
    DF['train']['x0'] = DF['train']['src'].drop(['Id', 'SalePrice'], axis=1)
    DF['train']['y'] = DF['train']['src'][['SalePrice']].copy()
    DF['train']['log_y'] = np.log1p(DF['train']['src']['SalePrice'].copy())

    DF['test']['id'] = DF['test']['src']['Id'].copy()
    DF['test']['x'] = DF['test']['src'].drop(['Id'], axis=1)


## Preprocessing

In [7]:
load_data()

num_cols = DF['train']['src'].select_dtypes(include=[np.number]).columns.tolist()
cat_cols = DF['train']['src'].select_dtypes(exclude=[np.number]).columns.tolist()

# Feature types:
# - Numerical continuous (e.g. GrLivArea, LotArea)
# - Numerical ordinal (e.g. OverallQual, OverallCond)
# - Categorical nominal (e.g. Neighborhood, RoofStyle)
# - Binary flags (e.g. CentralAir)
# - Missingness-as-information features (e.g. Alley)

# Structural categorical: missing means "feature absent"
struct_cat = [
    "Alley", "PoolQC", "Fence", "MiscFeature", "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
    "MasVnrType",
]

# Structural numeric: missing/NA means 0 amount (absence)
struct_num = [
    "MasVnrArea",
    "GarageCars", "GarageArea",
    "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
    "BsmtFullBath", "BsmtHalfBath",
]

true_cat = ["Electrical"]  # true missing
true_num = []              # keep empty for now

# Everything else
other_num = [c for c in num_cols if c not in struct_num + true_num]
other_cat = [c for c in cat_cols if c not in struct_cat + true_cat]

presence_map = {
    "GarageType": "HasGarage",
    "BsmtQual": "HasBasement",
    "FireplaceQu": "HasFireplace",
    "PoolQC": "HasPool",
}

skewed_num = [
    "LotArea", "LotFrontage", "MasVnrArea",
    "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
    "1stFlrSF", "2ndFlrSF", "GrLivArea",
    "GarageArea",
    "WoodDeckSF", "OpenPorchSF", "EnclosedPorch", "ScreenPorch", "3SsnPorch",
    "PoolArea", "MiscVal",
]

skewed_num = [c for c in skewed_num if c in DF['train']['src'].columns]  # safety

In [9]:
# Numeric pipelines (log happens before selection via FunctionTransformer)
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# For structural numeric (absence -> 0)
num_struct_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
    ("scaler", StandardScaler())
])

cat_struct_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="None")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = Pipeline([
    ("presence", PresenceIndicatorAdder(presence_map)),
    ("lotfrontage", LotFrontageByNeighborhoodImputer()),
    ("log1p", FunctionTransformer(transform.log1p_selected, validate=False)),
    ("ct", ColumnTransformer(
        transformers=[
            ("num_struct", num_struct_pipe, struct_num),
            ("num_other", num_pipe, other_num + list(presence_map.values()) + ["LotFrontage"]),
            ("cat_struct", cat_struct_pipe, struct_cat),
            ("cat_other", cat_pipe, other_cat),
            ("cat_true", cat_pipe, true_cat),
        ],
        remainder="drop"
    ))
])

## Feature Engineering

In [ ]:
load_data()

df_num = get_df_num(DF['train']['x0'])
df_num = date_transform(df_num)
df_num = binary_transform(df_num)
df_num = fill_missing(df_num)
df_num = drop_columns(df_num)
# print_null_cols(df_num)

df_test_num = get_df_num(DF['test']['x'])
df_test_num = date_transform(df_test_num)
df_test_num = binary_transform(df_test_num)
df_test_num = fill_missing(df_test_num)
df_test_num = drop_columns(df_test_num)
print_null_cols(df_test_num)

df_cat = get_df_cat(DF['train']['x0'])
df_test_cat = get_df_cat(DF['test']['x'])

df_all_cat_data = one_hot_encode(pd.concat([df_cat, df_test_cat], axis=0))
df_cat = df_all_cat_data[:len(DF['train']['x0'])]
df_test_cat = df_all_cat_data[len(DF['train']['x0']):]

DF['train']['x'] = pd.concat([df_num, df_cat], axis=1)
DF['test']['x'] = pd.concat([df_test_num, df_test_cat], axis=1)

Series([], dtype: float64)


#### Lasso CV

In [ ]:
DF['train']['x'] = scale(DF['train']['x'])

X_train, X_test, y_train, y_test = train_test_split(
    DF['train']['x'], np.log1p(DF['train']['y']), test_size=0.2, random_state=42
)

m_lasso_cv = train_lasso_cv(X_train, y_train)
sfm = SelectFromModel(m_lasso_cv, prefit=True)
print("Selected Features:", DF['train']['x'].columns[sfm.get_support()])

X_train_selected = sfm.transform(X_train)
X_test_selected = sfm.transform(X_test)

C:\Users\gqttr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\linear_model\_coordinate_descent.py:1756: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Selected Features: Index(['MSSubClass', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt',
       'YearRemodAdd', 'BsmtFinSF1', 'TotalBsmtSF', 'GrLivArea',
       'BsmtFullBath', 'FullBath', 'KitchenAbvGr', 'TotRmsAbvGrd',
       'Fireplaces', 'GarageCars', 'GarageArea', 'WoodDeckSF', 'ScreenPorch',
       'MSZoning_RM', 'LotShape_IR2', 'LotShape_IR3', 'LotConfig_CulDSac',
       'Neighborhood_ClearCr', 'Neighborhood_Crawfor', 'Neighborhood_Edwards',
       'Neighborhood_NridgHt', 'Neighborhood_StoneBr', 'Condition1_Norm',
       'Condition2_PosN', 'BldgType_Twnhs', 'Exterior1st_BrkComm',
       'Exterior1st_BrkFace', 'ExterQual_TA', 'Foundation_PConc',
       'BsmtCond_TA', 'BsmtExposure_Gd', 'BsmtExposure_No', 'BsmtFinType1_GLQ',
       'BsmtFinType1_Unf', 'Heating_Grav', 'CentralAir_Y', 'KitchenQual_TA',
       'Functional_Maj2', 'Functional_Sev', 'Functional_Typ', 'FireplaceQu_Gd',
       'FireplaceQu_nan', 'GarageType_Attchd', 'PavedDrive_Y', 'PoolQC_Gd',
       'SaleType_New',

C:\Users\gqttr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(
C:\Users\gqttr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but SelectFromModel was fitted without feature names
  warnings.warn(


#### Decision Tree

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    DF['train']['x'], DF['train']['y'], test_size=0.2, random_state=42
)

# DTR_grid_search(X_train, y_train)
# Best Parameters: {'criterion': 'absolute_error', 'max_depth': 15, 'min_samples_leaf': 16, 'min_samples_split': 2}
# Best Estimator: DecisionTreeRegressor(criterion='absolute_error', max_depth=15, min_samples_leaf=16)

m_clr =  DecisionTreeRegressor(criterion='absolute_error', max_depth=15, min_samples_leaf=16)
m_clr.fit(X_train, y_train)
print(model_performance(m_clr, X_test, y_test))

selected_features = get_important_features(m_clr, X_train)
X_train_selected = X_train[selected_features]
X_test_selected = X_test[selected_features]

m_clr_selected = DecisionTreeRegressor(criterion='absolute_error', max_depth=15, min_samples_leaf=16)
m_clr_selected.fit(X_train_selected, y_train)
print(model_performance(m_clr_selected, X_test_selected, y_test))


      MSSubClass  LotFrontage   LotArea  OverallQual  OverallCond  YearBuilt  \
892    -0.872563     0.006190 -0.210750    -0.071836     2.179628   0.273836   
1105    0.073375     1.277754  0.174303     1.374795    -0.517200  -0.752907   
413    -0.636078    -0.629592 -0.156028    -0.795151     0.381743   1.466183   
522    -0.163109    -0.902070 -0.552908    -0.071836     1.280685   0.803768   
1036   -0.872563     0.869037  0.238646     2.098110    -0.517200  -1.183477   
...          ...          ...       ...          ...          ...        ...   
479    -0.636078    -0.902070 -0.460202    -1.518467     1.280685   1.134975   
1361   -0.872563     2.458491  0.565370     0.651479    -0.517200  -1.117235   
802     0.073375    -0.311701 -0.232297     0.651479    -0.517200  -1.117235   
651     0.309859    -0.447940 -0.143601    -1.518467    -0.517200   1.035613   
722    -0.872563     0.006190 -0.240215    -1.518467     1.280685   0.041991   

      YearRemodAdd  MasVnrArea  BsmtFin

ValueError: Unable to coerce to Series, length must be 1: given 292

## Model Evaluations

#### Linear Regression

In [ ]:
df_num = scale(df_num)
X_train, X_test, y_train, y_test = train_test_split(
    df_num, np.log1p(DF['train']['y']), test_size=0.2, random_state=42
)
m_linreg = LinearRegression()
m_linreg.fit(X_train, y_train)
print(perf.model_performance(m_linreg, X_test, y_test))


      MSSubClass  LotFrontage   LotArea  OverallQual  OverallCond  YearBuilt  \
254    -0.872563     0.006190 -0.212153    -0.795151     0.381743   0.472560   
1066    0.073375    -0.493353 -0.268578    -0.071836     1.280685  -0.719786   
638    -0.636078    -0.130049 -0.174369    -0.795151     1.280685   2.029235   
799    -0.163109    -0.447940 -0.332419    -0.795151     1.280685   1.134975   
380    -0.163109    -0.902070 -0.552908    -0.795151     0.381743   1.565545   
...          ...          ...       ...          ...          ...        ...   
1095   -0.872563     0.369494 -0.120249    -0.071836    -0.517200  -1.150356   
1130   -0.163109    -0.220875 -0.271885    -1.518467    -2.315085   1.433062   
1294   -0.872563    -0.447940 -0.235003    -0.795151     1.280685   0.538802   
860    -0.163109    -0.675005 -0.288121     0.651479     2.179628   1.764269   
1126    1.492282    -0.765831 -0.684800     0.651479    -0.517200  -1.183477   

      YearRemodAdd  MasVnrArea  BsmtFin

#### Random Forest

In [ ]:
# RFR_grid_search(X_train_selected, y_train)
# Best Parameters: {'bootstrap': True, 'max_depth': 20, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}
# Best Estimator: RandomForestRegressor(max_depth=20, min_samples_leaf=2, n_estimators=200)

# m_rfr = RandomForestRegressor(max_depth=20, min_samples_leaf=2, n_estimators=200)
# m_rfr.fit(X_train, y_train)
# print(model_performance(m_rfr, X_test, y_test))

# m_rfr_selected = RandomForestRegressor(max_depth=20, min_samples_leaf=2, n_estimators=200)
# m_rfr_selected.fit(X_train_selected, y_train)
# print(model_performance(m_rfr_selected, X_test_selected, y_test))

## Model Training

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    DF['train']['x'], DF['train']['y'], test_size=0.2, random_state=42
)

m_xgb = xgboost.XGBRegressor()
m_xgb.fit(X_train, y_train)
print(model_performance(m_xgb, X_test, y_test))

# m_xgb_selected = xgboost.XGBRegressor()
# m_xgb_selected.fit(X_train_selected, y_train)
# print(model_performance(m_xgb_selected, X_test_selected, y_test))

KeyError: 'x'

## Results

In [ ]:
model = m_linreg
results = model.predict(df_test_num)
# results = np.expm1(model.predict(scale(df_test_num)))
print_results(results, DF['test']['id'], 'Id,SalePrice')